In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VjAxZSEJru3p0Pij8zYu")
project = rf.workspace("mehdis-workspace-jupje").project("smart-football-object-detection-apwnj")
version = project.version(1)
dataset = version.download("yolov8")


Defaulting to user installation because normal site-packages is not writeable
loading Roboflow workspace...
loading Roboflow project...


NotADirectoryError: [WinError 267] Nom de répertoire non valide: 'Smart-Football:-Object-Detection-1'

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import yaml

# Lire le fichier généré par Roboflow
with open(f"{dataset.location}/data.yaml", 'r') as stream:
    data = yaml.safe_load(stream)
    print(data)
# print("Nombre de classes (nc) :", data['nc'])
# print("Noms des classes :", data['names'])

{'names': ['Ball', 'Keeper', 'Player', 'Ref'], 'nc': 4, 'roboflow': {'license': 'CC BY 4.0', 'project': 'smart-football-object-detection-apwnj', 'url': 'https://universe.roboflow.com/mehdis-workspace-jupje/smart-football-object-detection-apwnj/dataset/1', 'version': 1, 'workspace': 'mehdis-workspace-jupje'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [14]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/41.8 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00


In [15]:
# Exemple de ce que vous feriez juste après :
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/best.pt")  # Charge un modèle YOLOv8 natif
model.train(data=f"{dataset.location}/data.yaml", epochs=100)  # Lance l'entraînement !

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Smart-Football:-Object-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7996163f3680>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
resultat  = model("./frame_1.jpg")
resultat[0].show()


image 1/1 D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\frame_1.jpg: 384x640 2 Balls, 1 Keeper, 20 Players, 2 Refs, 220.3ms
Speed: 10.5ms preprocess, 220.3ms inference, 12.4ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
# 2. Lancer la validation exclusivement sur le jeu de test
metrics = model.val(split='test')

# 3. Afficher les résultats clés
print(f"mAP50-95 (Précision globale) : {metrics.box.map:.4f}")
print(f"mAP50 (Précision à seuil 0.5) : {metrics.box.map50:.4f}")
print(f"Précision par classe : {metrics.box.mp:.4f}")
print(f"Rappel (Recall) par classe : {metrics.box.mr:.4f}")

# Afficher les performances détaillées par classe
for i, name in enumerate(metrics.names.values()):
    print(f"--- Classe : {name} ---")
    print(f"  Précision (Precision) : {metrics.box.class_result(i)[0]:.4f}")
    print(f"  Rappel (Recall)       : {metrics.box.class_result(i)[1]:.4f}")
    print(f"  mAP50                 : {metrics.box.class_result(i)[2]:.4f}")
    print("-" * 25)

Ultralytics 8.4.60  Python-3.13.2 torch-2.12.0+cpu CPU (Intel Core i5-1035G1 1.00GHz)
val: Fast image access  (ping: 4.69.4 ms, read: 1.10.4 MB/s, size: 46.1 KB)
val: Scanning D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 64.1it/s 0.2s
val: New cache created: D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.9s/it 2.9s
                   all         14        218      0.885      0.781      0.842      0.425
                  Ball         14         14      0.852      0.415      0.476      0.201
                Keeper          9          9      0.805      0.778      0.915      0.407
                Player         14        169      0.948     

In [2]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\train\weights\best.pt")
cap = cv2.VideoCapture(r"./WhatsApp Video 2026-07-01 at 02.22.30.mp4")

# Récupérer les propriétés de la vidéo d'origine
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Créer le fichier de sortie
out = cv2.VideoWriter(r".\resultat_annote.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    resultats = model(frame)
    frame_annotee = resultats[0].plot()

    out.write(frame_annotee)        # écrit la frame dans le fichier vidéo
    cv2.imshow("Detection", frame_annotee)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 320x640 2 Keepers, 15 Players, 1 Ref, 136.2ms
Speed: 5.5ms preprocess, 136.2ms inference, 14.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 Keepers, 15 Players, 2 Refs, 72.5ms
Speed: 4.4ms preprocess, 72.5ms inference, 4.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 Keepers, 14 Players, 1 Ref, 46.4ms
Speed: 2.3ms preprocess, 46.4ms inference, 8.5ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 3 Keepers, 14 Players, 1 Ref, 49.6ms
Speed: 1.6ms preprocess, 49.6ms inference, 3.7ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 13 Players, 1 Ref, 49.2ms
Speed: 1.9ms preprocess, 49.2ms inference, 3.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 Keepers, 14 Players, 1 Ref, 56.1ms
Speed: 2.2ms preprocess, 56.1ms inference, 5.6ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 3 Keepers, 13 Players, 1 Ref, 77.1ms
Speed: 14.3ms preprocess, 77.1ms inference, 4.7ms postprocess per image 